# 03 — App Data Preparation

## Goal

Prepare Streamlit-ready datasets for Dashboard 1: EU Structural Tradeoff Explorer.

This notebook does not perform advanced modeling, forecasting, policy recommendation, or causal analysis.

It prepares clean, interpretable tables for exploratory structural comparison across European economies.

## Guiding Questions

1. Can the master dataset support Dashboard 1?
2. Can we add time-period labels for shock and transition analysis?
3. Can we create reusable tables for Streamlit?
4. Which indicators require caution because of missingness or proxy limitations?

## Outputs

This notebook will create:

- `dashboard1_country_year.csv`
- later: `dashboard1_country_profiles.csv`
- later: `dashboard1_tradeoff_points.csv`

## Decision Checkpoint

Continue only if the exported table remains interpretable, transparent, and suitable for exploratory Dashboard 1 use.

Dashboard 2 remains evidence-gated.

In [1]:
import pandas as pd
import numpy as np

# Load main master dataset
df = pd.read_csv("../data/processed/eu_master_plus.csv")

# Basic check
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

print("\nYear range:")
print(df["year"].min(), "to", df["year"].max())

print("\nNumber of countries:")
print(df["country"].nunique())

Shape: (324, 21)
Columns:
['country', 'year', 'gdp_growth', 'inflation', 'unemployment', 'gini', 'education', 'rnd', 'ict_specialists', 'renewables', 'emissions', 'debt', 'defense_spending', 'economic_affairs_spending', 'fuel_energy_spending', 'communication_spending', 'environment_spending', 'health_spending', 'education_spending', 'social_protection_spending', 'total_gov_expenditure']

Year range:
2014 to 2025

Number of countries:
27


# Structural Time Period Framework

To support shock analysis, resilience exploration, and Streamlit filtering,
the dataset is augmented with structural transition period labels.

These labels are exploratory and descriptive only.

They help organize macro-structural phases across Europe:

- 2014–2019 → pre-shock baseline
- 2020–2021 → COVID transition
- 2022–2025 → energy / geopolitical transition

The purpose is interpretability and comparative exploration,
not causal attribution.

In [2]:
# ============================================================
# Structural period labels
# ============================================================

def assign_period_label(year):

    if 2014 <= year <= 2019:
        return "pre_shock_baseline"

    elif 2020 <= year <= 2021:
        return "covid_transition"

    elif 2022 <= year <= 2025:
        return "energy_geopolitical_transition"

    else:
        return "outside_scope"


# Main period label
df["period_label"] = df["year"].apply(assign_period_label)


# Boolean flags
df["is_pre_shock"] = (
    df["period_label"] == "pre_shock_baseline"
)

df["is_covid_transition"] = (
    df["period_label"] == "covid_transition"
)

df["is_energy_transition"] = (
    df["period_label"]
    == "energy_geopolitical_transition"
)


# Quick validation
df[
    [
        "country",
        "year",
        "period_label",
        "is_pre_shock",
        "is_covid_transition",
        "is_energy_transition"
    ]
].head(15)

,country,year,period_label,is_pre_shock,is_covid_transition,is_energy_transition
0,AT,2014,pre_shock_baseline,True,False,False
1,AT,2015,pre_shock_baseline,True,False,False
2,AT,2016,pre_shock_baseline,True,False,False
3,AT,2017,pre_shock_baseline,True,False,False
4,AT,2018,pre_shock_baseline,True,False,False
5,AT,2019,pre_shock_baseline,True,False,False
6,AT,2020,covid_transition,False,True,False
7,AT,2021,covid_transition,False,True,False
8,AT,2022,energy_geopolitical_transition,False,False,True
9,AT,2023,energy_geopolitical_transition,False,False,True


# Dataset Reliability & Coverage Assessment

Before creating normalized indicators and Streamlit exports,
the dataset is evaluated for reliability and completeness.

This step is important because macroeconomic indicators across Europe
have varying levels of availability, reporting consistency,
and methodological limitations.

The purpose is not to reject imperfect indicators,
but to identify:

- which KPIs are robust enough for exploratory comparison,
- which require caution,
- and where missingness may bias interpretation.

This project remains exploratory and uncertainty-aware.

In [3]:
# ============================================================
# Missingness summary by KPI
# ============================================================

# KPI columns only
kpi_columns = [

    "gdp_growth",
    "inflation",
    "unemployment",
    "gini",
    "education",
    "rnd",
    "ict_specialists",
    "renewables",
    "emissions",
    "debt",

    "defense_spending",
    "economic_affairs_spending",
    "fuel_energy_spending",
    "communication_spending",
    "environment_spending",
    "health_spending",
    "education_spending",
    "social_protection_spending",
    "total_gov_expenditure"
]


# Build missingness summary
missingness_summary = pd.DataFrame({

    "kpi": kpi_columns,

    "missing_count": [
        df[col].isna().sum()
        for col in kpi_columns
    ],

    "missing_percent": [
        round(df[col].isna().mean() * 100, 1)
        for col in kpi_columns
    ],

    "non_missing_count": [
        df[col].notna().sum()
        for col in kpi_columns
    ]
})


# Sort by highest missingness
missingness_summary = (
    missingness_summary
    .sort_values(
        by="missing_percent",
        ascending=False
    )
)


# Display
missingness_summary

,kpi,missing_count,missing_percent,non_missing_count
0,gdp_growth,0,0.0,324
1,inflation,0,0.0,324
2,unemployment,0,0.0,324
3,gini,0,0.0,324
4,education,0,0.0,324
5,rnd,0,0.0,324
6,ict_specialists,0,0.0,324
7,renewables,0,0.0,324
8,emissions,0,0.0,324
9,debt,0,0.0,324


# KPI Interpretation & Reliability Notes

Although the integrated dataset is fully populated after cleaning and controlled filling procedures,
the indicators still represent simplified structural proxies.

Some values may include:

- interpolation,
- smoothing,
- harmonization across reporting systems,
- or cautious forward-filling for recent years.

The project therefore focuses on:

- structural comparison,
- exploratory interpretation,
- and educational tradeoff understanding,

rather than precise econometric inference.

Indicators should be interpreted as directional macro-structural signals,
not exact representations of complex socioeconomic systems.

In [7]:
# ============================================================
# KPI metadata table with dimensions, units, and interpretation
# ============================================================

kpi_metadata = pd.DataFrame({

    "kpi": [
        "gdp_growth",
        "inflation",
        "unemployment",
        "gini",
        "education",
        "rnd",
        "ict_specialists",
        "renewables",
        "emissions",
        "debt",

        "defense_spending",
        "economic_affairs_spending",
        "fuel_energy_spending",
        "communication_spending",
        "environment_spending",
        "health_spending",
        "education_spending",
        "social_protection_spending",
        "total_gov_expenditure"
    ],

    "dimension": [
        "economy",
        "economy",
        "economy",
        "society",
        "society",
        "innovation",
        "innovation",
        "sustainability",
        "sustainability",
        "fiscal_resilience",

        "public_investment",
        "public_investment",
        "public_investment",
        "public_investment",
        "public_investment",
        "public_investment",
        "public_investment",
        "public_investment",
        "public_investment"
    ],

    "unit": [
        "% annual change",        # gdp_growth
        "% annual change",        # inflation
        "% of labor force",       # unemployment
        "Gini coefficient",       # gini
        "% of population",        # education
        "% of GDP",               # rnd
        "% of employment",        # ict_specialists
        "% of energy use",        # renewables
        "index / emissions proxy",# emissions
        "% of GDP",               # debt

        "% of GDP",               # defense_spending
        "% of GDP",               # economic_affairs_spending
        "% of GDP",               # fuel_energy_spending
        "% of GDP",               # communication_spending
        "% of GDP",               # environment_spending
        "% of GDP",               # health_spending
        "% of GDP",               # education_spending
        "% of GDP",               # social_protection_spending
        "% of GDP"                # total_gov_expenditure
    ],

    "higher_is_generally": [
        "positive",
        "contextual",
        "negative",
        "negative",
        "positive",
        "positive",
        "positive",
        "positive",
        "negative",
        "contextual",

        "contextual",
        "contextual",
        "contextual",
        "contextual",
        "positive",
        "positive",
        "positive",
        "positive",
        "contextual"
    ],

    "interpretation_risk": [
        "medium",
        "medium",
        "medium",
        "high",
        "medium",
        "medium",
        "medium",
        "medium",
        "high",
        "high",

        "medium",
        "medium",
        "high",
        "high",
        "medium",
        "medium",
        "medium",
        "medium",
        "medium"
    ],

    "short_description": [
        "Annual GDP growth rate",
        "Consumer price inflation",
        "Unemployment rate",
        "Income inequality indicator",
        "Educational attainment level",
        "Research and development intensity",
        "ICT specialist workforce share",
        "Renewable energy share",
        "Greenhouse gas emissions indicator",
        "Government debt level",

        "Defense expenditure share",
        "Economic affairs expenditure",
        "Fuel and energy expenditure",
        "Communication expenditure",
        "Environmental protection expenditure",
        "Health expenditure",
        "Education expenditure",
        "Social protection expenditure",
        "Total government expenditure"
    ]
})

kpi_metadata

,kpi,dimension,unit,higher_is_generally,interpretation_risk,short_description
0,gdp_growth,economy,% annual change,positive,medium,Annual GDP growth rate
1,inflation,economy,% annual change,contextual,medium,Consumer price inflation
2,unemployment,economy,% of labor force,negative,medium,Unemployment rate
3,gini,society,Gini coefficient,negative,high,Income inequality indicator
4,education,society,% of population,positive,medium,Educational attainment level
5,rnd,innovation,% of GDP,positive,medium,Research and development intensity
6,ict_specialists,innovation,% of employment,positive,medium,ICT specialist workforce share
7,renewables,sustainability,% of energy use,positive,medium,Renewable energy share
8,emissions,sustainability,index / emissions proxy,negative,high,Greenhouse gas emissions indicator
9,debt,fiscal_resilience,% of GDP,contextual,high,Government debt level


# KPI Semantic Metadata

To support interpretability and educational exploration,
each KPI is annotated with simplified semantic metadata.

This includes:

- short descriptions,
- directional interpretation,
- and whether higher values are generally interpreted positively, negatively, or contextually.

Importantly:

These interpretations are not universal truths.

Many macroeconomic indicators involve structural tradeoffs,
historical context,
and competing policy objectives.

The metadata is therefore designed for exploratory understanding,
not normative judgment.

In [8]:
# ============================================================
# KPI semantic metadata
# ============================================================

kpi_metadata["higher_is_generally"] = [

    "positive",   # gdp_growth
    "contextual", # inflation
    "negative",   # unemployment

    "negative",   # gini
    "positive",   # education

    "positive",   # rnd
    "positive",   # ict_specialists

    "positive",   # renewables
    "negative",   # emissions

    "contextual", # debt

    "contextual", # defense_spending
    "contextual", # economic_affairs_spending
    "contextual", # fuel_energy_spending
    "contextual", # communication_spending
    "positive",   # environment_spending
    "positive",   # health_spending
    "positive",   # education_spending
    "positive",   # social_protection_spending
    "contextual"  # total_gov_expenditure
]


kpi_metadata["short_description"] = [

    "Annual GDP growth rate",
    "Consumer price inflation",
    "Unemployment rate",

    "Income inequality indicator",
    "Educational attainment level",

    "Research and development intensity",
    "ICT specialist workforce share",

    "Renewable energy share",
    "Greenhouse gas emissions",

    "Government debt level",

    "Defense expenditure share",
    "Economic affairs expenditure",
    "Fuel and energy expenditure",
    "Communication expenditure",
    "Environmental protection expenditure",
    "Health expenditure",
    "Education expenditure",
    "Social protection expenditure",
    "Total government expenditure"
]


kpi_metadata

,kpi,dimension,unit,higher_is_generally,interpretation_risk,short_description
0,gdp_growth,economy,% annual change,positive,medium,Annual GDP growth rate
1,inflation,economy,% annual change,contextual,medium,Consumer price inflation
2,unemployment,economy,% of labor force,negative,medium,Unemployment rate
3,gini,society,Gini coefficient,negative,high,Income inequality indicator
4,education,society,% of population,positive,medium,Educational attainment level
5,rnd,innovation,% of GDP,positive,medium,Research and development intensity
6,ict_specialists,innovation,% of employment,positive,medium,ICT specialist workforce share
7,renewables,sustainability,% of energy use,positive,medium,Renewable energy share
8,emissions,sustainability,index / emissions proxy,negative,high,Greenhouse gas emissions
9,debt,fiscal_resilience,% of GDP,contextual,high,Government debt level


In [9]:
# ============================================================
# Streamlit helper columns
# ============================================================

country_name_map = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "HR": "Croatia",
    "CY": "Cyprus",
    "CZ": "Czechia",
    "DK": "Denmark",
    "EE": "Estonia",
    "FI": "Finland",
    "FR": "France",
    "DE": "Germany",
    "EL": "Greece",
    "HU": "Hungary",
    "IE": "Ireland",
    "IT": "Italy",
    "LV": "Latvia",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "MT": "Malta",
    "NL": "Netherlands",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "SK": "Slovakia",
    "SI": "Slovenia",
    "ES": "Spain",
    "SE": "Sweden"
}

period_order_map = {
    "pre_shock_baseline": 1,
    "covid_transition": 2,
    "energy_geopolitical_transition": 3,
    "outside_scope": 99
}

df["country_name"] = df["country"].map(country_name_map)
df["period_order"] = df["period_label"].map(period_order_map)

latest_year = df["year"].max()
df["is_latest_year"] = df["year"] == latest_year

df[
    [
        "country",
        "country_name",
        "year",
        "period_label",
        "period_order",
        "is_latest_year"
    ]
].head(15)

,country,country_name,year,period_label,period_order,is_latest_year
0,AT,Austria,2014,pre_shock_baseline,1,False
1,AT,Austria,2015,pre_shock_baseline,1,False
2,AT,Austria,2016,pre_shock_baseline,1,False
3,AT,Austria,2017,pre_shock_baseline,1,False
4,AT,Austria,2018,pre_shock_baseline,1,False
5,AT,Austria,2019,pre_shock_baseline,1,False
6,AT,Austria,2020,covid_transition,2,False
7,AT,Austria,2021,covid_transition,2,False
8,AT,Austria,2022,energy_geopolitical_transition,3,False
9,AT,Austria,2023,energy_geopolitical_transition,3,False


# KPI Group Definitions

To support Streamlit filtering and structural exploration,
KPIs are grouped into conceptual macro-structural dimensions.

These are conceptual analytical groupings,
not statistically-derived latent factors.

The dimensions may later evolve based on EDA findings.

In [11]:
# ============================================================
# KPI group definitions
# ============================================================

economy_kpis = [
    "gdp_growth",
    "inflation",
    "unemployment"
]

society_kpis = [
    "gini",
    "education"
]

innovation_kpis = [
    "rnd",
    "ict_specialists"
]

sustainability_kpis = [
    "renewables",
    "emissions"
]

fiscal_kpis = [
    "debt",
    "total_gov_expenditure"
]

public_investment_kpis = [

    "defense_spending",
    "economic_affairs_spending",
    "fuel_energy_spending",
    "communication_spending",
    "environment_spending",
    "health_spending",
    "education_spending",
    "social_protection_spending"
]


# Combined list
all_kpis = (

    economy_kpis
    + society_kpis
    + innovation_kpis
    + sustainability_kpis
    + fiscal_kpis
    + public_investment_kpis
)


print("Total KPIs:", len(all_kpis))

print("\nEconomy:")
print(economy_kpis)

print("\nSociety:")
print(society_kpis)

print("\nInnovation:")
print(innovation_kpis)

print("\nSustainability:")
print(sustainability_kpis)

print("\nFiscal:")
print(fiscal_kpis)

print("\nPublic investment:")
print(public_investment_kpis)

Total KPIs: 19

Economy:
['gdp_growth', 'inflation', 'unemployment']

Society:
['gini', 'education']

Innovation:
['rnd', 'ict_specialists']

Sustainability:
['renewables', 'emissions']

Fiscal:
['debt', 'total_gov_expenditure']

Public investment:
['defense_spending', 'economic_affairs_spending', 'fuel_energy_spending', 'communication_spending', 'environment_spending', 'health_spending', 'education_spending', 'social_protection_spending']


# Export Streamlit App Tables

The first app-ready outputs are exported for Dashboard 1.

These files separate:

- country-year analytical data
- KPI interpretation metadata

This keeps the Streamlit architecture clean and maintainable.

In [12]:
# ============================================================
# Export first Streamlit-ready tables
# ============================================================

from pathlib import Path

# Create app data folder if needed
app_data_dir = Path("../data/app")
app_data_dir.mkdir(parents=True, exist_ok=True)

# Main Dashboard 1 country-year table
dashboard1_country_year = df.copy()

# Export files
dashboard1_country_year.to_csv(
    app_data_dir / "dashboard1_country_year.csv",
    index=False
)

kpi_metadata.to_csv(
    app_data_dir / "kpi_metadata.csv",
    index=False
)

print("Export complete:")
print(app_data_dir / "dashboard1_country_year.csv")
print(app_data_dir / "kpi_metadata.csv")

print("\nDashboard 1 table shape:", dashboard1_country_year.shape)
print("KPI metadata shape:", kpi_metadata.shape)

Export complete:
..\data\app\dashboard1_country_year.csv
..\data\app\kpi_metadata.csv

Dashboard 1 table shape: (324, 28)
KPI metadata shape: (19, 6)


# Export Validation

Before continuing the analytical pipeline,
the exported Streamlit tables are reloaded and validated.

This ensures:

- files were saved correctly,
- columns are preserved,
- paths work properly,
- and the app architecture remains stable.

In [13]:
# ============================================================
# Validate exported app tables
# ============================================================

# Reload exported tables
country_year_check = pd.read_csv(
    "../data/app/dashboard1_country_year.csv"
)

kpi_metadata_check = pd.read_csv(
    "../data/app/kpi_metadata.csv"
)


# Basic validation
print("COUNTRY-YEAR TABLE")
print("------------------")
print("Shape:", country_year_check.shape)

print("\nColumns:")
print(country_year_check.columns.tolist())


print("\n\nKPI METADATA TABLE")
print("------------------")
print("Shape:", kpi_metadata_check.shape)

print("\nColumns:")
print(kpi_metadata_check.columns.tolist())

COUNTRY-YEAR TABLE
------------------
Shape: (324, 28)

Columns:
['country', 'year', 'gdp_growth', 'inflation', 'unemployment', 'gini', 'education', 'rnd', 'ict_specialists', 'renewables', 'emissions', 'debt', 'defense_spending', 'economic_affairs_spending', 'fuel_energy_spending', 'communication_spending', 'environment_spending', 'health_spending', 'education_spending', 'social_protection_spending', 'total_gov_expenditure', 'period_label', 'is_pre_shock', 'is_covid_transition', 'is_energy_transition', 'country_name', 'period_order', 'is_latest_year']


KPI METADATA TABLE
------------------
Shape: (19, 6)

Columns:
['kpi', 'dimension', 'unit', 'higher_is_generally', 'interpretation_risk', 'short_description']


# EU Long-Horizon Normalization

The first normalization framework compares countries relative to the overall European structural baseline across the full analysis horizon (2014–2025).

This normalization supports:

- long-term structural positioning,
- comparative country profiling,
- archetype exploration,
- and high-level tradeoff interpretation.

Interpretation example:

A positive z-score means a country is above the long-term EU average for that indicator across the full observation period.

This normalization is exploratory and comparative only.

In [15]:
# ============================================================
# EU long-horizon normalization
# ============================================================

# Create normalized copy
dashboard1_country_year_norm = dashboard1_country_year.copy()


# Apply EU-wide long-horizon z-score normalization
for kpi in all_kpis:

    new_col = f"{kpi}_eu_long_zscore"

    mean_value = dashboard1_country_year_norm[kpi].mean()
    std_value = dashboard1_country_year_norm[kpi].std()

    dashboard1_country_year_norm[new_col] = (
        dashboard1_country_year_norm[kpi] - mean_value
    ) / std_value


# Quick validation
zscore_cols = [
    col
    for col in dashboard1_country_year_norm.columns
    if col.endswith("_eu_long_zscore")
]


print("Created z-score columns:", len(zscore_cols))

print("\nExample columns:")
print(zscore_cols[:5])


dashboard1_country_year_norm[
    [
        "country",
        "year",
        "gdp_growth",
        "gdp_growth_eu_long_zscore",
        "renewables",
        "renewables_eu_long_zscore"
    ]
].head(15)

Created z-score columns: 19

Example columns:
['gdp_growth_eu_long_zscore', 'inflation_eu_long_zscore', 'unemployment_eu_long_zscore', 'gini_eu_long_zscore', 'education_eu_long_zscore']


,country,year,gdp_growth,gdp_growth_eu_long_zscore,renewables,renewables_eu_long_zscore
0,AT,2014,0.8,-0.489308,33.550,0.803761
1,AT,2015,1.3,-0.351594,33.497,0.799464
2,AT,2016,2.1,-0.131253,33.370,0.789170
3,AT,2017,2.3,-0.076167,33.136,0.770201
4,AT,2018,2.5,-0.021082,33.784,0.822729
5,AT,2019,1.8,-0.213881,33.755,0.820378
6,AT,2020,-6.3,-2.444840,36.545,1.046539
7,AT,2021,4.9,0.639943,34.643,0.892361
8,AT,2022,5.3,0.750114,34.057,0.844859
9,AT,2023,-0.8,-0.929991,41.600,1.456303


# EU-Year Normalization

This normalization compares each country to the EU distribution within the same year.

For each KPI and year:

(country value - EU mean in that year) / EU standard deviation in that year

This supports:

- yearly country comparison,
- tradeoff scatterplots,
- shock-period exploration,
- and Dashboard 1 interactive filtering.

This is different from EU long-horizon normalization.

EU-long normalization shows broad structural identity.
EU-year normalization shows relative positioning within each year.

In [16]:
# ============================================================
# EU-year normalization
# ============================================================

for kpi in all_kpis:

    new_col = f"{kpi}_eu_year_zscore"

    yearly_mean = dashboard1_country_year_norm.groupby("year")[kpi].transform("mean")
    yearly_std = dashboard1_country_year_norm.groupby("year")[kpi].transform("std")

    dashboard1_country_year_norm[new_col] = (
        dashboard1_country_year_norm[kpi] - yearly_mean
    ) / yearly_std


# Quick validation
eu_year_zscore_cols = [
    col
    for col in dashboard1_country_year_norm.columns
    if col.endswith("_eu_year_zscore")
]


print("Created EU-year z-score columns:", len(eu_year_zscore_cols))

print("\nExample columns:")
print(eu_year_zscore_cols[:5])


dashboard1_country_year_norm[
    [
        "country",
        "year",
        "gdp_growth",
        "gdp_growth_eu_year_zscore",
        "renewables",
        "renewables_eu_year_zscore"
    ]
].head(15)

Created EU-year z-score columns: 19

Example columns:
['gdp_growth_eu_year_zscore', 'inflation_eu_year_zscore', 'unemployment_eu_year_zscore', 'gini_eu_year_zscore', 'education_eu_year_zscore']


,country,year,gdp_growth,gdp_growth_eu_year_zscore,renewables,renewables_eu_year_zscore
0,AT,2014,0.8,-0.618019,33.550,1.177635
1,AT,2015,1.3,-0.513951,33.497,1.114129
2,AT,2016,2.1,-0.391178,33.370,1.101288
3,AT,2017,2.3,-0.648208,33.136,1.026809
4,AT,2018,2.5,-0.461101,33.784,1.049904
5,AT,2019,1.8,-0.801215,33.755,0.950644
6,AT,2020,-6.3,-0.630425,36.545,1.061275
7,AT,2021,4.9,-0.836698,34.643,0.798158
8,AT,2022,5.3,0.654505,34.057,0.652993
9,AT,2023,-0.8,-0.728651,41.600,1.103031


# Intermediate Export — EU Normalization Layer

The dataset is exported after implementing:

- EU long-horizon normalization
- EU-year normalization

This creates a stable intermediate analytical layer
before adding country-relative normalization.

In [17]:
# ============================================================
# Export intermediate normalized dataset
# ============================================================

dashboard1_country_year_norm.to_csv(
    "../data/app/dashboard1_country_year_norm.csv",
    index=False
)

print("Export complete:")
print("../data/app/dashboard1_country_year_norm.csv")

print("\nShape:")
print(dashboard1_country_year_norm.shape)

Export complete:
../data/app/dashboard1_country_year_norm.csv

Shape:
(324, 66)


# Country-Relative Long-Horizon Normalization

This normalization evaluates each country relative to its own historical baseline across 2014–2025.

For each KPI:

(country value - country historical mean)
/
(country historical standard deviation)

This normalization supports:

- transition analysis,
- internal structural evolution,
- resilience interpretation,
- and trajectory exploration.

Interpretation example:

A positive z-score means the country is above its own long-term historical average for that indicator.

This differs from EU-relative normalization,
which compares countries to Europe rather than to themselves.

In [18]:
# ============================================================
# Country-relative long-horizon normalization
# ============================================================

for kpi in all_kpis:

    new_col = f"{kpi}_country_zscore"

    country_mean = (
        dashboard1_country_year_norm
        .groupby("country")[kpi]
        .transform("mean")
    )

    country_std = (
        dashboard1_country_year_norm
        .groupby("country")[kpi]
        .transform("std")
    )

    dashboard1_country_year_norm[new_col] = (
        dashboard1_country_year_norm[kpi] - country_mean
    ) / country_std


# Quick validation
country_zscore_cols = [

    col
    for col in dashboard1_country_year_norm.columns
    if col.endswith("_country_zscore")
]


print("Created country-relative z-score columns:", len(country_zscore_cols))

print("\nExample columns:")
print(country_zscore_cols[:5])


dashboard1_country_year_norm[
    [
        "country",
        "year",
        "gdp_growth",
        "gdp_growth_country_zscore",
        "renewables",
        "renewables_country_zscore"
    ]
].head(15)

Created country-relative z-score columns: 19

Example columns:
['gdp_growth_country_zscore', 'inflation_country_zscore', 'unemployment_country_zscore', 'gini_country_zscore', 'education_country_zscore']


,country,year,gdp_growth,gdp_growth_country_zscore,renewables,renewables_country_zscore
0,AT,2014,0.8,-0.116708,33.550,-0.660332
1,AT,2015,1.3,0.050018,33.497,-0.673775
2,AT,2016,2.1,0.316779,33.370,-0.705990
3,AT,2017,2.3,0.383469,33.136,-0.765345
4,AT,2018,2.5,0.450159,33.784,-0.600976
5,AT,2019,1.8,0.216743,33.755,-0.608332
6,AT,2020,-6.3,-2.484212,36.545,0.099370
7,AT,2021,4.9,1.250442,34.643,-0.383085
8,AT,2022,5.3,1.383823,34.057,-0.531728
9,AT,2023,-0.8,-0.650230,41.600,1.381604


# Final Normalized Dashboard 1 Table

The fully normalized Dashboard 1 analytical table is exported after implementing:

- EU long-horizon normalization
- EU-year normalization
- country-relative normalization

This table becomes the main analytical backbone for:

- Streamlit Dashboard 1,
- comparative exploration,
- tradeoff visualization,
- and future derived analytical tables.

In [20]:
# ============================================================
# Export final normalized Dashboard 1 table
# ============================================================

dashboard1_country_year_norm.to_csv(
    "../data/app/dashboard1_country_year_full_norm.csv",
    index=False
)

print("Export complete:")
print("../data/app/dashboard1_country_year_full_norm.csv")

print("\nShape:")
print(dashboard1_country_year_norm.shape)

Export complete:
../data/app/dashboard1_country_year_full_norm.csv

Shape:
(324, 85)


# Unified Normalization Reference Statistics

To support interpretability, reproducibility, and Streamlit explanations,
all normalization reference statistics are exported into a unified metadata table.

The table stores:

- normalization type,
- grouping context,
- KPI name,
- mean,
- and standard deviation.

This supports:

- tooltip explanations,
- methodological transparency,
- validation,
- and future dashboard extensions.

In [21]:
# ============================================================
# Unified normalization reference statistics
# ============================================================

reference_stats = []


# ------------------------------------------------------------
# EU long-horizon stats
# ------------------------------------------------------------

for kpi in all_kpis:

    reference_stats.append({

        "normalization_type": "eu_long",

        "group_key": "all_years",

        "kpi": kpi,

        "mean": df[kpi].mean(),

        "std": df[kpi].std()
    })


# ------------------------------------------------------------
# EU yearly stats
# ------------------------------------------------------------

for kpi in all_kpis:

    yearly_stats = (

        df
        .groupby("year")[kpi]
        .agg(["mean", "std"])
        .reset_index()
    )

    for _, row in yearly_stats.iterrows():

        reference_stats.append({

            "normalization_type": "eu_year",

            "group_key": row["year"],

            "kpi": kpi,

            "mean": row["mean"],

            "std": row["std"]
        })


# ------------------------------------------------------------
# Country historical stats
# ------------------------------------------------------------

for kpi in all_kpis:

    country_stats = (

        df
        .groupby("country")[kpi]
        .agg(["mean", "std"])
        .reset_index()
    )

    for _, row in country_stats.iterrows():

        reference_stats.append({

            "normalization_type": "country_long",

            "group_key": row["country"],

            "kpi": kpi,

            "mean": row["mean"],

            "std": row["std"]
        })


# Convert to dataframe
normalization_reference_stats = pd.DataFrame(reference_stats)


# Preview
print("Shape:")
print(normalization_reference_stats.shape)

print("\nNormalization types:")
print(
    normalization_reference_stats[
        "normalization_type"
    ].unique()
)

normalization_reference_stats.head(15)

Shape:
(760, 5)

Normalization types:
<StringArray>
['eu_long', 'eu_year', 'country_long']
Length: 3, dtype: str


,normalization_type,group_key,kpi,mean,std
0,eu_long,all_years,gdp_growth,2.576543,3.630726
1,eu_long,all_years,inflation,2.702469,3.452472
2,eu_long,all_years,unemployment,7.296605,3.883762
3,eu_long,all_years,gini,29.637654,3.919310
4,eu_long,all_years,education,77.078086,9.119515
5,eu_long,all_years,rnd,1.679198,0.892119
6,eu_long,all_years,ict_specialists,4.413580,1.464291
7,eu_long,all_years,renewables,23.634512,12.336367
8,eu_long,all_years,emissions,8.100000,2.696770
9,eu_long,all_years,debt,68.509568,37.994367


In [22]:
# ============================================================
# Export normalization reference statistics
# ============================================================

normalization_reference_stats.to_csv(
    "../data/app/normalization_reference_stats.csv",
    index=False
)

print("Export complete:")
print("../data/app/normalization_reference_stats.csv")

print("\nShape:")
print(normalization_reference_stats.shape)

Export complete:
../data/app/normalization_reference_stats.csv

Shape:
(760, 5)


# Final Export Validation

Before continuing into advanced exploratory analysis,
all final exported analytical tables are reloaded and validated.

This ensures:

- export integrity,
- correct normalization structure,
- stable Streamlit architecture,
- and reproducible analytical foundations.

In [23]:
# ============================================================
# Final export validation
# ============================================================

# Reload exported files
full_norm_check = pd.read_csv(
    "../data/app/dashboard1_country_year_full_norm.csv"
)

reference_stats_check = pd.read_csv(
    "../data/app/normalization_reference_stats.csv"
)


# ------------------------------------------------------------
# FULL NORMALIZED TABLE
# ------------------------------------------------------------

print("FULL NORMALIZED TABLE")
print("---------------------")

print("Shape:")
print(full_norm_check.shape)

print("\nExample columns:")
print(full_norm_check.columns[:15].tolist())


# Count normalization columns
eu_long_cols = [
    c for c in full_norm_check.columns
    if c.endswith("_eu_long_zscore")
]

eu_year_cols = [
    c for c in full_norm_check.columns
    if c.endswith("_eu_year_zscore")
]

country_cols = [
    c for c in full_norm_check.columns
    if c.endswith("_country_zscore")
]


print("\nEU-long columns:", len(eu_long_cols))
print("EU-year columns:", len(eu_year_cols))
print("Country-relative columns:", len(country_cols))


# ------------------------------------------------------------
# REFERENCE STATS TABLE
# ------------------------------------------------------------

print("\n\nREFERENCE STATS TABLE")
print("---------------------")

print("Shape:")
print(reference_stats_check.shape)

print("\nNormalization types:")
print(
    reference_stats_check[
        "normalization_type"
    ].unique()
)

print("\nExample rows:")
print(reference_stats_check.head(10))

FULL NORMALIZED TABLE
---------------------
Shape:
(324, 85)

Example columns:
['country', 'year', 'gdp_growth', 'inflation', 'unemployment', 'gini', 'education', 'rnd', 'ict_specialists', 'renewables', 'emissions', 'debt', 'defense_spending', 'economic_affairs_spending', 'fuel_energy_spending']

EU-long columns: 19
EU-year columns: 19
Country-relative columns: 19


REFERENCE STATS TABLE
---------------------
Shape:
(760, 5)

Normalization types:
<StringArray>
['eu_long', 'eu_year', 'country_long']
Length: 3, dtype: str

Example rows:
  normalization_type  group_key              kpi       mean        std
0            eu_long  all_years       gdp_growth   2.576543   3.630726
1            eu_long  all_years        inflation   2.702469   3.452472
2            eu_long  all_years     unemployment   7.296605   3.883762
3            eu_long  all_years             gini  29.637654   3.919310
4            eu_long  all_years        education  77.078086   9.119515
5            eu_long  all_years  